# 01 — Data Preparation

Downloads PlantNet-300K (positive class) and Caltech-256 non-nature classes (negative class),
assigns binary labels, balances the dataset, and produces `train.csv`, `val.csv`, `test.csv`.

**Run on Kaggle Notebooks with GPU enabled.**

Before running:
- Add `plantnet/plantnet-300k` via Kaggle → Add Data
- Add `jessicali9530/caltech256` via Kaggle → Add Data

In [ ]:
import pathlib
import random
import pandas as pd
from sklearn.model_selection import train_test_split

# --- Positive class: PlantNet-300K ---
PLANT_DIR = pathlib.Path('/kaggle/input/plantnet-300k/plantnet_300K/images/train')
pos_images = list(PLANT_DIR.rglob('*.jpg'))
print(f'Positive images found: {len(pos_images)}')

In [ ]:
# --- Negative class: Caltech-256 non-nature categories ---
# These classes are clearly not nature elements
NON_NATURE_CLASSES = [
    'car-side-101', 'airplanes-101', 'motorbikes-101', 'bicycle-101',
    'boats-101', 'cellphone-101', 'laptop-101', 'refrigerator-101',
    'filing-cabinet-101', 'desk-globe-101', 'fire-hydrant-101', 'skateboard-101'
]
NEG_BASE = pathlib.Path('/kaggle/input/caltech256/256_ObjectCategories')
neg_images = [
    p
    for cls in NON_NATURE_CLASSES
    if (NEG_BASE / cls).exists()
    for p in (NEG_BASE / cls).rglob('*.jpg')
]
print(f'Negative images found: {len(neg_images)}')

In [ ]:
# --- Build balanced DataFrame ---
random.seed(42)
random.shuffle(neg_images)

# Sample negatives to match positive count
sample_size = min(len(pos_images), len(neg_images))
pos_rows = [{'path': str(p), 'label': 1} for p in pos_images[:sample_size]]
neg_rows = [{'path': str(p), 'label': 0} for p in neg_images[:sample_size]]

df = pd.DataFrame(pos_rows + neg_rows).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Total: {len(df)} images  |  Positive: {df.label.sum()}  |  Negative: {(df.label==0).sum()}')

In [ ]:
# --- Train / Val / Test split: 70% / 15% / 15% ---
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)

train_df.to_csv('/kaggle/working/train.csv', index=False)
val_df.to_csv('/kaggle/working/val.csv',     index=False)
test_df.to_csv('/kaggle/working/test.csv',   index=False)

print(f'Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}')